# PID control

**Hold 20 m/s, then follow the line.**

That sounds like the easiest job in the book, and the starter does neither.
Unit 1 measured it cruising at 19.49 m/s with a target of 20.
Unit 3 showed that 19.49 is no accident: it is the root of a quadratic, the speed at which a proportional speed controller's throttle exactly balances drag.
Unit 2 measured how far it wanders off the centreline, and the starter never even looks at that number: it steers at a waypoint three ahead.

This unit builds the controller that every racing stack starts from, the **PID** controller, and uses it twice: on the speed, and on the steering.
By the end of the lesson you will have:

- a way to measure any controller, the **step response** (overshoot, settling time, steady-state error);
- a discrete PID with **anti-windup**, tuned by its step response;
- a steering law built on Unit 2's instruments and Unit 3's model, with the **heading error as its D term** and a **curvature feedforward**;
- a submission, `agents/unit-04.py`, that holds 20 m/s and sits a quarter to a third closer to the line than the starter, beyond the noise, on all three tracks.

First, the problem, measured.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races, spread
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track
from racekit.variants import write_variant

AGENTS = Path("../../projects/racekit/agents")
STARTER = AGENTS / "unit-01.py"
UNIT_04 = AGENTS / "unit-04.py"


def rms_offset(result):
    # Unit 2's cross-track offset, RMS over the driving rows (respawn holds excluded)
    offset = result.telemetry["offset"][result.telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def max_offset(result):
    offset = result.telemetry["offset"][result.telemetry["held"] == 0]
    return float(np.max(np.abs(offset)))


starter = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
cruise = starter.telemetry["v"][starter.telemetry["t"] > 30.0]
print(starter)
print(f"the starter's speed after 30 s: median {np.median(cruise):.4f} m/s (its target is 20)")
print(f"its cross-track offset: RMS {rms_offset(starter):.4f} m, max {max_offset(starter):.3f} m")
assert starter.finished and starter.collisions == 0 and abs(starter.total_time - 223.50) < 1e-9
assert abs(np.median(cruise) - 19.49) < 0.005

Two errors, and the same job for both:

- the **speed error** e_v = 20 − v must go to zero and stay there;
- the **cross-track error** e (Unit 2: the signed distance from the centreline, left positive) must go to zero.

A feedback controller measures an error every tick and chooses an action that shrinks it.
The PID controller does it with three terms:

$$u(t) = K_P\,e(t) \;+\; K_I \int_0^t e(s)\,ds \;+\; K_D\,\frac{de}{dt}.$$

Each term has one job: **P** reacts to the error now, **I** to the error accumulated in the past, **D** to where the error is heading.
The car is controlled in ticks of h = 0.05 s, so the controller is discrete:

$$u_n = K_P\,e_n \;+\; K_I \sum_{k \le n} e_k\,h \;+\; K_D\,\frac{e_n - e_{n-1}}{h} \quad\text{(or } K_D \text{ times a measured rate of } e\text{)}.$$

§2–§4 take the terms one at a time on the speed, where the plant is one ODE (Unit 3's longitudinal map).
§5 brings in D, and §6 turns to the steering.

## 1. Measuring a controller: the step response

Ask a controller to take the car from rest to 20 m/s on a straight, and record the speed after every tick for 30 s.
Three numbers summarize the record:

- **overshoot**: how far the speed goes above the target, as a percentage of the step (20 − v₀);
- **settling time** (2% band): the first time from which the speed stays within 2% of the step of the target (±0.4 m/s here) for the rest of the run, or "never" if it is still outside at the end;
- **steady-state error**: the target minus the speed at the end of the run.

The car is racekit's kinematic car with jitter 0, advanced exactly as the runner advances it (10 substeps per 0.05 s tick), so every number below is deterministic.
A controller here is any object with an `update(error, dt)` method; the positive part of its command is the throttle and the negative part the brake, as in the starter.

In [ ]:
DT = CONTROL_DT  # 0.05 s: one control tick


def step_response(controller, v0=0.0, target=20.0, seconds=30.0, integrals=None):
    # Drive straight from v0 with the controller on the speed error; return the speed after every tick.
    # If an array is passed as `integrals`, it receives the controller's integral after every tick.
    car = KinematicCar()  # jitter 0: a deterministic run
    car.reset(0.0, 0.0, 0.0, v0)
    speeds = np.zeros(int(round(seconds / DT)))
    for n in range(len(speeds)):
        command = controller.update(target - car.v, DT)
        throttle = min(max(command, 0.0), 1.0)  # the positive part drives
        brake = min(max(-command, 0.0), 1.0)  # the negative part brakes
        car.advance(throttle, 0.0, brake, 0.0, 0, DT, 10)
        speeds[n] = car.v
        if integrals is not None:
            integrals[n] = controller.integral
    return speeds


def step_metrics(speeds, v0=0.0, target=20.0):
    # (overshoot in % of the step, settling time in s (math.inf = never), error at the end in m/s)
    step = target - v0
    times = DT * np.arange(1, len(speeds) + 1)
    overshoot = max(0.0, float(np.max(speeds) - target) / step * 100.0)
    outside = np.nonzero(np.abs(speeds - target) > 0.02 * step)[0]
    if len(outside) == 0:
        settling = 0.0
    elif outside[-1] == len(speeds) - 1:
        settling = math.inf
    else:
        settling = float(times[outside[-1] + 1])
    return overshoot, settling, float(target - speeds[-1])


def describe(name, metrics):
    overshoot, settling, error = metrics
    settle = "never" if settling == math.inf else f"{settling:.2f} s"
    print(f"{name:>30}: overshoot {overshoot:6.2f}%, settling {settle:>7}, error at the end {error:+.4f} m/s")

## 2. P, and why a P loop needs an error

The controller is a class, because the I and D terms need memory: the running integral and the previous error.
This first version is the plain discrete PID above.
With `rate=None` the D term is the finite difference (e_n − e_{n−1}) / h; §5 explains why you would rather pass a measured rate.

In [ ]:
class PID:
    # u = kp * e + ki * (integral of e) + kd * (rate of e), one update per control tick.

    def __init__(self, kp, ki=0.0, kd=0.0):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.integral = 0.0
        self.previous = None  # the previous error (for the finite-difference D term)

    def update(self, error, dt, rate=None):
        self.integral += error * dt
        output = self.kp * error + self.ki * self.integral
        if self.kd:
            if rate is None:
                rate = (error - self.previous) / dt if self.previous is not None else 0.0
            output += self.kd * rate
        self.previous = error
        return output


P_ROWS = {}
for kp in (0.05, 0.5, 2.0):
    P_ROWS[kp] = step_metrics(step_response(PID(kp)))
    describe(f"P {kp}" + (" (the starter)" if kp == 0.05 else ""), P_ROWS[kp])

assert all(row[0] == 0.0 for row in P_ROWS.values())
assert P_ROWS[0.05][1] == math.inf and abs(P_ROWS[0.05][2] - 0.5081) < 1e-4
assert abs(P_ROWS[0.5][1] - 3.60) < 1e-9 and abs(P_ROWS[0.5][2] - 0.0531) < 1e-4
assert abs(P_ROWS[2.0][1] - 3.30) < 1e-9 and abs(P_ROWS[2.0][2] - 0.0133) < 1e-4

The starter's gain never settles: it stops about 0.51 m/s short, outside the ±0.4 m/s band.
A forty-times larger gain shrinks the error to 0.013 m/s, but **no P gain removes it**.

The reason is the drag balance of Unit 3.
In the steady state v̇ = 0, so the throttle must exactly cancel the drag, a_T · throttle = c·v².
A P loop's throttle is K_P·e, and v = v* − e, so the error it settles at solves

$$a_T\,K_P\,e = c\,(v^* - e)^2 \quad\Longleftrightarrow\quad c\,e^2 - (2c\,v^* + a_T K_P)\,e + c\,v^{*2} = 0,$$

the smaller root.
Zero error would mean zero throttle, and zero throttle cannot hold any speed against drag.
For large K_P the root is about c·v*² / (a_T K_P): the error falls as 1/K_P and never reaches zero.

In [ ]:
A_THROTTLE, C_DRAG = 6.0, 4.0e-4  # Unit 3: the kinematic car's longitudinal map (a model, not the truth)


def p_equilibrium_error(kp, target=20.0):
    # The smaller root of C e^2 - (2 C v* + a_T kp) e + C v*^2 = 0: where a P loop settles.
    b = 2.0 * C_DRAG * target + A_THROTTLE * kp
    return (b - math.sqrt(b * b - 4.0 * C_DRAG * C_DRAG * target * target)) / (2.0 * C_DRAG)


for kp in (0.05, 0.5, 2.0):
    print(f"P {kp:4}: the drag balance predicts e = {p_equilibrium_error(kp):.4f} m/s; after 30 s: {P_ROWS[kp][2]:.4f} m/s")

time_constant = 1.0 / (A_THROTTLE * 0.05 + 2.0 * C_DRAG * 19.49)
print(f"the starter settles at {20.0 - p_equilibrium_error(0.05):.4f} m/s; its loop's time constant is {time_constant:.2f} s")
assert abs(p_equilibrium_error(0.05) - 0.5067) < 1e-4 and abs(20.0 - p_equilibrium_error(0.05) - 19.49) < 0.005
assert abs(p_equilibrium_error(0.5) - P_ROWS[0.5][2]) < 1e-4 and abs(p_equilibrium_error(2.0) - P_ROWS[2.0][2]) < 1e-4

**19.49 m/s is the starter's P loop at its drag balance**: Unit 1's number and Unit 3's quadratic, now seen from the error's side.
The 30 s figure (0.5081) is still 0.0014 m/s above the equilibrium (0.5067) because the starter's loop is slow:
linearize v̇ = a_T K_P (v* − v) − c v² around the equilibrium and the error decays like e^(−t/τ) with τ = 1 / (a_T K_P + 2cv) ≈ 3.2 s, so after 30 s about e^(−9.4) of the start is left.
The larger gains reach their equilibrium within 1e-4 m/s.

To remove the error, the controller needs a term that can hold a throttle when the error is zero.

## 3. I, and windup from a standing start

The integral term K_I ∫ e dt grows as long as any error remains.
So the loop can only come to rest where e = 0, and there the I term alone holds the drag-balancing throttle.
Here is the PI controller (K_P, K_I) = (0.5, 0.1), from rest and from 15 m/s.

In [ ]:
SECONDS = 30.0
plain_from_rest = np.zeros(int(round(SECONDS / DT)))
speeds_plain = step_response(PID(0.5, 0.1), integrals=plain_from_rest)
PI_PLAIN = step_metrics(speeds_plain)
PI_PLAIN_15 = step_metrics(step_response(PID(0.5, 0.1), v0=15.0), v0=15.0)
describe("PI (0.5, 0.1) from rest", PI_PLAIN)
describe("PI (0.5, 0.1) from 15 m/s", PI_PLAIN_15)
print(f"the integral peaks at {np.max(plain_from_rest):.2f} m (the I term then asks for {0.1 * np.max(plain_from_rest):.2f})")

assert abs(PI_PLAIN[0] - 28.71) < 0.01 and abs(PI_PLAIN[1] - 17.70) < 1e-9 and abs(PI_PLAIN[2] + 0.0290) < 1e-4
assert abs(PI_PLAIN_15[0] - 7.19) < 0.01 and abs(PI_PLAIN_15[1] - 8.40) < 1e-9

The steady-state error is gone, but from rest the speed overshoots by **29%**, to almost 26 m/s, and takes 17.7 s to settle.
That is **integrator windup**.

From rest, the error is 20 m/s and the P term alone asks for a command of 10: the throttle is **saturated** at 1 for the first few seconds, whatever the controller says.
The car accelerates as fast as it can, and the integral keeps adding the large error the whole time.
By the time the car reaches 20 m/s, the integral holds about 34 m (m/s × s) and the I term alone asks for 3.4, more than three times full throttle.
To unwind it, the error must go negative for a long time: that is the overshoot.
The integral was integrating an error that the controller **could not do anything about**.

The fix used here is **conditional integration**: integrate a tick's error only if the command with it would be inside the range the actuator can follow, or if the error drives the command back towards that range.
For the speed loop the range is the **throttle's, (0, 1)**: a command at or above 1 (full throttle) or at or below 0 (no throttle: coasting or braking) freezes the integral, unless the error points back into the band.
So while the throttle is pinned at 1 the integral stops growing, and while the loop brakes it does not wind down below the coasting point.
The class below is the same PID with an optional `limits=(low, high)` and that rule; it is exactly the `PID` class in `agents/unit-04.py`.

In [ ]:
class PID:
    # u = kp * e + ki * (integral of e) + kd * (rate of e), with conditional-integration anti-windup.
    # limits=(low, high) is the range the actuator can follow; update() returns the unclipped command.

    def __init__(self, kp, ki=0.0, kd=0.0, limits=None):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.limits = limits
        self.reset()

    def reset(self):
        self.integral = 0.0
        self.previous = None

    def update(self, error, dt, rate=None):
        candidate = self.integral + error * dt
        command = self.kp * error + self.ki * candidate
        if self.limits is None:
            self.integral = candidate
        else:
            low, high = self.limits
            inside = low < command < high
            unwinding = (command >= high and error < 0) or (command <= low and error > 0)
            if inside or unwinding:
                self.integral = candidate
        output = self.kp * error + self.ki * self.integral
        if self.kd:
            if rate is None:
                rate = (error - self.previous) / dt if self.previous is not None else 0.0
            output += self.kd * rate
        self.previous = error
        return output


clamped_from_rest = np.zeros(int(round(SECONDS / DT)))
speeds_clamped = step_response(PID(0.5, 0.1, limits=(0.0, 1.0)), integrals=clamped_from_rest)
PI_CLAMPED = step_metrics(speeds_clamped)
PI_CLAMPED_15 = step_metrics(step_response(PID(0.5, 0.1, limits=(0.0, 1.0)), v0=15.0), v0=15.0)
describe("anti-windup PI from rest", PI_CLAMPED)
describe("anti-windup PI from 15 m/s", PI_CLAMPED_15)
print(f"the integral now peaks at {np.max(clamped_from_rest):.2f} m")

assert abs(PI_CLAMPED[0] - 0.26) < 0.01 and abs(PI_CLAMPED[1] - 3.55) < 1e-9 and abs(PI_CLAMPED[2] + 0.0003) < 1e-4
assert abs(PI_CLAMPED_15[0] - 0.98) < 0.01 and abs(PI_CLAMPED_15[1] - 1.35) < 1e-9
assert np.max(clamped_from_rest) < np.max(plain_from_rest)

t = DT * np.arange(1, len(speeds_plain) + 1)
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
top.plot(t, step_response(PID(0.5)), label="P 0.5")
top.plot(t, speeds_plain, label="PI (0.5, 0.1), no anti-windup")
top.plot(t, speeds_clamped, label="PI (0.5, 0.1), conditional integration")
top.axhline(20.0, color="grey", linestyle="--", linewidth=1.0)
top.set_ylabel("speed (m/s)")
top.set_title("Step response from rest to 20 m/s (the kinematic car, jitter 0)")
top.legend()
bottom.plot(t, plain_from_rest, label="no anti-windup", color="C1")
bottom.plot(t, clamped_from_rest, label="conditional integration", color="C2")
bottom.set_xlabel("t (s)")
bottom.set_ylabel("integral of e (m)")
bottom.legend()
plt.show()

| PI (0.5, 0.1) | overshoot | settling (2% band) | error at 30 s |
|---|---|---|---|
| from rest, no anti-windup | 28.71% | 17.70 s | −0.0290 m/s |
| from rest, conditional integration | 0.26% | 3.55 s | −0.0003 m/s |
| from 15 m/s, no anti-windup | 7.19% | 8.40 s | |
| from 15 m/s, conditional integration | 0.98% | 1.35 s | |

The integral now holds little more than the steady state needs (the drag-balancing throttle c·v²/a_T ≈ 0.027 at 20 m/s, an integral of about 0.27 m with K_I = 0.1), and the overshoot almost disappears.
Even the small step from 15 m/s winds up without the rule: the P term alone asks for 2.5 there, so the throttle saturates there too.
Windup is not a corner case: **every actuator saturates**, and every integral you write needs a rule for it.

## 4. Tuning by step response

The step response turns tuning into measurement.
Here is a small grid of (K_P, K_I) with conditional integration, from rest and from 15 m/s.

In [ ]:
GRID = {}
print("  kp    ki | from rest: overshoot, settling | from 15 m/s: overshoot, settling")
for kp in (0.25, 0.5, 1.0):
    for ki in (0.05, 0.1, 0.2):
        big = step_metrics(step_response(PID(kp, ki, limits=(0.0, 1.0))))
        small = step_metrics(step_response(PID(kp, ki, limits=(0.0, 1.0)), v0=15.0), v0=15.0)
        GRID[(kp, ki)] = (big, small)
        print(f"{kp:4} {ki:5} | {big[0]:6.2f}%  {big[1]:5.2f} s | {small[0]:6.2f}%  {small[1]:5.2f} s")
assert GRID[(0.5, 0.1)][0][:2] == PI_CLAMPED[:2]

Read the grid the way an engineer would.

- **From rest, the gains barely matter**: every row settles in 3.35–7.1 s, because for the first three seconds the throttle is saturated and the car accelerates at its limit whatever the controller asks.
  A saturated step measures the car, not the controller.
- **The small step discriminates.**
  A small K_P with a large K_I is slow and overshoots (K_P 0.25, K_I 0.2: 15.7%); the I term then does the P term's job, late.
- On this model the step response alone would pick K_P = 1.0 (no overshoot, 1.05 s from 15 m/s).
  The reference submission keeps **(0.5, 0.1)**: 0.26% and 0.98% overshoot, and settling within 0.3 s of the best.

Is that 0.3 s worth anything in a race?
A race has exactly one standing start, so race the two gains and see.

In [ ]:
folder = Path(mkdtemp())
stiffer = write_variant(UNIT_04, folder / "speed_kp_1.py", {"SPEED_KP": "1.0"})
reference = run_race(UNIT_04, "oval", car=KinematicCar(jitter=0.0))
stiff = run_race(stiffer, "oval", car=KinematicCar(jitter=0.0))
print(f"SPEED_KP = 0.5 (the reference): {reference.total_time:.2f} s;  SPEED_KP = 1.0: {stiff.total_time:.2f} s")
assert abs(reference.total_time - 216.35) < 1e-9 and abs(stiff.total_time - reference.total_time) < 1e-9

The same time to the tick.
The step response is the right **instrument** for a controller, but the objective is the race: tune what the race time can feel.
Two more cautions apply to every gain in this unit.
These are the **kinematic car's** gains: on CARLA the throttle's effect depends on gear and engine speed, and Unit 8 re-identifies the car before any gain is trusted there.
And a higher P gain also reacts harder to every bump of noise, which no deterministic step can show.

## 5. D, from a measured rate

The D term reacts to the **trend**: K_D · de/dt.
The speed loop does not need one (its plant is first order, and PI already settles in 3.55 s), but the steering loop does.
Here is why, on the simplest steering problem there is.

A straight line along +x, the car driving along it at a constant 20 m/s (Unit 3's drag-balancing throttle c·v²/a_T), and a cross-track error e = y − y_line, left positive.
The heading error is ψ_e = (line heading) − ψ = −ψ.
Differentiate (Unit 3's model):

$$\dot e = \dot y = v\sin\psi = -v\sin\psi_e, \qquad \ddot e = v\cos\psi\;\dot\psi = \frac{v^2\cos\psi\,\tan\delta}{L} \approx \frac{v^2}{L}\,\delta.$$

The wheel angle sets the error's **acceleration**, not its rate.
A P law δ = −K_E·e then gives ë ≈ −(v²K_E/L)·e: a spring with no damper, an oscillation that never decays.
The damper is a term in ė: δ = −K_E·e − K_D·ė.
There are two ways to get ė:

- the **finite difference** (e_n − e_{n−1}) / h, which every textbook PID uses;
- a **measured rate**: ė = −v sin ψ_e, from the speed and the heading error, which the car measures every tick.

With the measured rate the D term is K_D·v·sin ψ_e ≈ K_D·v·ψ_e: **the heading error is the D term**.
Race both, from 1 m off the line.

In [ ]:
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1; a positive steer turns right
K_E = 0.2  # rad of wheel angle per m of cross-track error
V = 20.0  # m/s
K_D = 1.0 / V  # s: K_HEAD / v with the reference's K_HEAD = 1.0 (see section 6)


def lane_drive(d_term, start_offset=0.0, change_at=math.inf, seconds=20.0):
    # Hold V along the line y = line_y (0, or 1 m after `change_at` seconds) with
    # delta = -K_E * e + D term; d_term is "none", "difference" or "measured".
    # Returns the cross-track error and the wheel angle at every tick.
    car = KinematicCar()
    car.reset(0.0, start_offset, 0.0, V)
    hold = C_DRAG * V * V / A_THROTTLE
    ticks = int(round(seconds / DT))
    errors, angles = np.zeros(ticks), np.zeros(ticks)
    previous = None
    for n in range(ticks):
        line_y = 1.0 if n * DT >= change_at else 0.0
        e = car.y - line_y
        psi_e = 0.0 - car.yaw  # the line's heading minus the car's
        if d_term == "difference":
            d = 0.0 if previous is None else -K_D * (e - previous) / DT
        elif d_term == "measured":
            d = -K_D * (-car.v * math.sin(psi_e))  # -K_D * e_dot, with e_dot = -v sin(psi_e)
        else:
            d = 0.0
        previous = e
        delta = -K_E * e + d
        car.advance(hold, min(max(-delta / DELTA_MAX, -1.0), 1.0), 0.0, 0.0, 0, DT, 10)
        errors[n], angles[n] = e, delta
    return errors, angles


RECOVERY = {}
for d_term in ("none", "difference", "measured"):
    errors, _ = lane_drive(d_term, start_offset=1.0)
    RECOVERY[d_term] = errors
    print(f"{d_term:>10}: max |e| over the last 5 s = {np.max(np.abs(errors[-100:])):.2e} m; deepest crossing {-np.min(errors):.3f} m")
assert np.max(np.abs(RECOVERY["none"][-100:])) > 1.0  # P alone: the oscillation grows
assert np.max(np.abs(RECOVERY["difference"][-100:])) < 1e-6 and np.max(np.abs(RECOVERY["measured"][-100:])) < 1e-6

Without D the car weaves, and in discrete time the weave **grows**: after 20 s it swings by metres.
With either D term it settles onto the line within a few seconds.
So far the two D terms look the same.
Now move the line: at t = 3 s the line jumps 1 m to the left (a lane change, or in a speed loop a new target speed).

In [ ]:
KICK = {}
for d_term in ("difference", "measured"):
    errors, angles = lane_drive(d_term, change_at=3.0, seconds=8.0)
    KICK[d_term] = (errors, angles)
    print(f"{d_term:>10}: largest |wheel angle| = {np.max(np.abs(angles)):.3f} rad ({math.degrees(np.max(np.abs(angles))):.0f} degrees)")
assert abs(np.max(np.abs(KICK["difference"][1])) - (K_E * 1.0 + K_D * 1.0 / DT)) < 1e-9  # 0.2 + 1.0
assert abs(np.max(np.abs(KICK["measured"][1])) - K_E * 1.0) < 1e-9

t = DT * np.arange(len(KICK["measured"][0]))
fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
for d_term, (errors, angles) in KICK.items():
    top.plot(t, errors, label={"difference": "D from the finite difference", "measured": "D from the measured rate"}[d_term])
    bottom.plot(t, angles)
top.set_ylabel("cross-track error e (m)")
top.set_title("The line jumps 1 m left at t = 3 s")
top.legend()
bottom.set_xlabel("t (s)")
bottom.set_ylabel("wheel angle (rad)")
plt.show()

That spike is the **derivative kick**.
When the set point jumps, e jumps by 1 m in one tick, and the finite difference turns the jump into a rate of 1 / 0.05 = 20 m/s: the D term asks for K_D · 20 = 1.0 rad, and the wheel angle hits 1.2 rad, **69°**, for one tick, at 20 m/s.
The measured rate never sees the jump, because the car did not move sideways: the heading did not change, so ė = −v sin ψ_e did not either.
The kick is the visible failure; the quiet one is noise.
A finite difference divides every measurement error by h = 0.05 s, so it multiplies noise by 20.

Rule: **take D from a measured rate when you have one.**
For the cross-track error you always do: it is the heading error.

## 6. Steering on the track

On the track the line is Unit 2's polyline, and nothing above changes except the frame.
At each tick:

- e is Unit 2's cross-track error, measured on the segment i = `segment_index` that the windowed projection picks (left positive);
- the segment's heading is θ_i = atan2(y_{i+1} − y_i, x_{i+1} − x_i), and the **heading error** is ψ_e = wrap(θ_i − ψ) (Unit 2's `wrap_angle`);
- in the segment's frame the derivation of §5 is unchanged, so ė = −v sin ψ_e.

The reference submission's steering law is

$$\delta = -K_E\,e \;+\; K_{HEAD}\,\psi_e \;+\; \arctan(L\,\kappa_i), \qquad \text{steer} = \operatorname{clip}\!\left(-\frac{\delta}{\delta_{\max}},\,-1,\,1\right),$$

with K_E = 0.2 rad/m and K_HEAD = 1.0.
It is a PD law on the error −e whose D term is the measured heading error: K_HEAD·ψ_e ≈ K_D·v·sin ψ_e, so **K_HEAD = K_D·v**.
That has a consequence worth stating now: with K_HEAD fixed, the effective damping K_D = K_HEAD / v **falls as 1/v**, from 0.05 s at 20 m/s to 0.033 s at 30 m/s, a third less.
Fixed gains tuned at one speed are softer at a higher one; Unit 5 meets this.

The last term is the **feedforward**.
On a corner of curvature κ the car holds the line only with the wheel angle of Unit 3's steady circle, δ = arctan(L·κ).
The PD terms can make that angle only from an error: without the feedforward, a steady corner needs −K_E·e = arctan(L·κ), an offset of e = −arctan(L·κ)/K_E.
With it, the PD terms only correct what the feedforward gets wrong.
κ_i is Unit 2's Menger curvature at segment i, computed once in `__init__`.
Here is the offset the feedforward saves on each track's corners:

In [ ]:
def menger_curvature(xs, ys):
    # Unit 2: the signed curvature at each waypoint from (i - 1, i, i + 1), cyclic; left positive.
    n = len(xs)
    before, after = (np.arange(n) - 1) % n, (np.arange(n) + 1) % n
    ax, ay, cx, cy = xs[before], ys[before], xs[after], ys[after]
    twice_area = (xs - ax) * (cy - ay) - (ys - ay) * (cx - ax)
    sides = np.hypot(xs - ax, ys - ay) * np.hypot(cx - xs, cy - ys) * np.hypot(cx - ax, cy - ay)
    return 2.0 * twice_area / sides


CORNER_OFFSET = {}
for name in ("oval", "test-circuit", "monza-like"):
    track = load_track(name)
    kappa = np.abs(menger_curvature(track.x, track.y))
    feedforward = np.array([math.atan2(WHEELBASE * k, 1.0) for k in kappa])  # arctan(L kappa)
    CORNER_OFFSET[name] = feedforward / K_E
    print(f"{name:>12}: tightest R = {1.0 / np.max(kappa):6.2f} m, its feedforward {np.max(feedforward):.4f} rad;"
          f" without it the PD law holds that corner {np.max(CORNER_OFFSET[name]):.2f} m off the line")
assert 0.7 < np.max(CORNER_OFFSET["test-circuit"]) < 0.9

Most of a metre on `test-circuit`'s tightest corner, from the controller's structure alone, before any noise.

Now race the reference and its two ablations against the starter on `test-circuit`, with racekit's actuator jitter (0.05) and seed 0.
`write_variant` (Unit 1) makes each ablation a real submission: `FEEDFORWARD = False` removes the feedforward, and `K_HEAD = 0.0` removes the D term.
The no-D race is capped at 400 s.

In [ ]:
no_feedforward = write_variant(UNIT_04, folder / "no_feedforward.py", {"FEEDFORWARD": "False"})
no_d_term = write_variant(UNIT_04, folder / "no_d_term.py", {"K_HEAD": "0.0"})
ENTRIES = {
    "starter": (STARTER, 1800.0),
    "unit 4": (UNIT_04, 1800.0),
    "unit 4, no feedforward": (no_feedforward, 1800.0),
    "unit 4, no D term": (no_d_term, 400.0),
}
SEED_0 = {}
for name, (submission, cap) in ENTRIES.items():
    result = run_race(submission, "test-circuit", car=KinematicCar(jitter=0.05), seed=0, max_seconds=cap)
    SEED_0[name] = result
    print(f"{name:>24}: {result.total_time:7.2f} s, {result.collisions:2d} collisions, "
          f"RMS offset {rms_offset(result):.4f} m, max {max_offset(result):.3f} m")

for name in ("starter", "unit 4", "unit 4, no feedforward"):
    assert SEED_0[name].finished and SEED_0[name].collisions == 0
assert rms_offset(SEED_0["unit 4"]) < rms_offset(SEED_0["starter"]) < rms_offset(SEED_0["unit 4, no feedforward"])
assert not SEED_0["unit 4, no D term"].finished and SEED_0["unit 4, no D term"].collisions == 30

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
for name in ("starter", "unit 4, no feedforward", "unit 4"):
    telemetry = SEED_0[name].telemetry
    window = (telemetry["t"] >= 20.0) & (telemetry["t"] <= 80.0)
    ax.plot(telemetry["t"][window], telemetry["offset"][window], label=name)
ax.axhline(0.0, color="grey", linewidth=1.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("cross-track offset (m, left positive)")
ax.set_title("test-circuit, jitter 0.05, seed 0: 60 s of the first lap")
ax.legend()
plt.show()

Each term earns its place.

- **Without the D term** the car weaves off the track 30 times in 400 s and never finishes: the spring with no damper from §5, on a real track.
- **Without the feedforward** the car finishes, but it holds every corner off the line, as predicted, and on `test-circuit` it is **worse than the starter**.
- **With both**, the offset is a small noise around zero: the remaining error is the jitter the PD terms correct.

Measured on five seeds (0–4, jitter 0.05), the RMS offset medians are:

| track | Unit 4 | no feedforward | starter |
|---|---|---|---|
| oval | 0.1041 m | 0.1327 m | 0.1397 m |
| test-circuit | 0.1048 m | 0.1794 m | 0.1586 m |
| monza-like | 0.1029 m | 0.1374 m | 0.1416 m |

and without the D term (on `test-circuit`, capped at 400 s) every seed crashes, with 30, 32, 31, 29 and 30 collisions.

## 7. The scorecard: Unit 4 against the starter

Unit 1's rule: a claim that A beats B needs at least five noisy runs and a gap larger than the spread of either.
Here are five seeds on `oval`, for the RMS offset and for the race time.

In [ ]:
def jittered():
    return KinematicCar(jitter=0.05)


SCORE = {}
for name, submission in (("unit 4", UNIT_04), ("starter", STARTER)):
    runs = repeat_races(submission, "oval", n=5, car_factory=jittered)
    rms = [rms_offset(result) for result in runs.results]
    SCORE[name] = (runs, rms)
    print(f"{name:>8}: RMS offset median {np.median(rms):.4f} m (range {spread(rms):.4f}); "
          f"time median {runs.median:.2f} s (spread {runs.spread:.2f}); collisions {runs.collisions}")

(u4_runs, u4_rms), (st_runs, st_rms) = SCORE["unit 4"], SCORE["starter"]
assert u4_runs.collisions == 0 and st_runs.collisions == 0 and u4_runs.all_finished and st_runs.all_finished
assert abs(np.median(u4_rms) - 0.1041) < 5e-4 and abs(spread(u4_rms) - 0.0062) < 5e-4
assert abs(np.median(st_rms) - 0.1397) < 5e-4 and abs(spread(st_rms) - 0.0099) < 5e-4
assert abs(u4_runs.median - 216.40) < 0.05 + 1e-9 and abs(st_runs.median - 220.75) < 0.05 + 1e-9
assert np.median(st_rms) - np.median(u4_rms) > max(spread(u4_rms), spread(st_rms))
assert noise_aware_better(u4_runs.times, st_runs.times)
print(f"race time gained: {100.0 * (st_runs.median - u4_runs.median) / st_runs.median:.2f}%")

Both claims hold beyond the noise on `oval`.
On all three tracks (five seeds each, jitter 0.05, all 30 races clean):

| track | RMS offset, Unit 4 | starter | time median (spread), Unit 4 | starter |
|---|---|---|---|---|
| oval | 0.1041 m (range 0.0062) | 0.1397 m (0.0099) | 216.40 s (0.05) | 220.75 s (0.35) |
| test-circuit | 0.1048 m (0.0050) | 0.1586 m (0.0055) | 400.60 s (0.10) | 407.50 s (0.65) |
| monza-like | 0.1029 m (0.0042) | 0.1416 m (0.0040) | 858.55 s (0.00) | 870.50 s (0.45) |

On every track the RMS gap exceeds the larger range, and Unit 4 is faster beyond the noise: by 2.0%, 1.7% and 1.4% of the race time.
With jitter 0 the reference runs 216.35 / 400.55 / 858.45 s, all clean.

Where does the time come from?
Almost all of it from the speed loop: 20 m/s instead of the starter's 19.49 (with jitter, about 19.8: the throttle noise is clipped at 0, so on average it pushes, which is why the jittered starter is faster than the deterministic one; the PI loop's integral removes that bias along with drag's).
Following the line more closely buys accuracy, not time, because the target speed is the same.
Going faster is Units 6 and 7.

## 8. What a controller that only reacts cannot do

Look at what the Unit 4 law uses: the error **now**, the heading **now**, and the curvature of the segment the car is **on**.
Nothing in it looks down the road.
On racekit's kinematic car that is enough, because the car does exactly what it is told, the moment it is told.

A real car does not.
Commands take time to reach the wheels, and every tick of delay turns a correction into a late correction.
A late D term no longer damps; it can push the weave along.
And with fixed gains the damping K_D = K_HEAD / v is already a third softer at 30 m/s than at 20.

A human driver does not steer at the road under the bonnet.
They look ahead, at where the car will be, and steer toward it.
Unit 5 builds that controller, **pure pursuit**, and measures the trade it makes: stability under delay, against cutting corners when there is none.